# Partie E - Fine-tuning mono-tache et multi-tache



## 0. Environnement et regles du TP

Le decoupage officiel train/test est conserve. Le test officiel ne sert qu'a l'evaluation finale. Toutes les decisions sont prises sur la validation.

In [8]:
# Configuration environnement GPU & patch de compatibilite PyTorch < 2.6
RANDOM_STATE = 42

import os
import re
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Peripherique :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

# Contournement securite torch < 2.6 pour charger .bin sans changer de driver
import transformers.utils.import_utils as iu
import transformers.modeling_utils as mu
iu.check_torch_load_is_safe = lambda: None
mu.check_torch_load_is_safe = lambda: None

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, average_precision_score

def _column(frame, names):
    for name in names:
        if name in frame.columns:
            return name
    raise KeyError("Colonne introuvable : " + str(names))

RESULTS = []

def add_result(name, y_true, scores, threshold=0.5, split="validation"):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores)
    pred = (scores >= threshold).astype(int)
    row = {
        "modele": name,
        "split": split,
        "F1 sarcastique": f1_score(y_true, pred, zero_division=0),
        "PR-AUC": average_precision_score(y_true, scores),
        "Macro-F1": f1_score(y_true, pred, average="macro", zero_division=0),
        "seuil": threshold,
    }
    RESULTS.append(row)
    return row


Peripherique : cuda
GPU : NVIDIA GeForce RTX 2050


In [9]:
BASE = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/"
full = pd.read_csv(BASE + "training_data.csv")
test = pd.read_csv(BASE + "testing_data.csv")
full["id"] = range(len(full))
test["id"] = range(len(test))
full["text_clean"] = full["tweet"].fillna("").astype(str).map(lambda s: re.sub(r"\s+", " ", s).strip())
test["text_clean"] = test["tweet"].fillna("").astype(str).map(lambda s: re.sub(r"\s+", " ", s).strip())

train_df, val_df = train_test_split(
    full, test_size=0.10, random_state=RANDOM_STATE, stratify=full["sarcasm"]
)
print("Train officiel :", full.shape, "| Train split :", len(train_df), "| Val split :", len(val_df))
thresholds = np.arange(0.1, 0.91, 0.05)
display(full.head(3))


Train officiel : (12548, 6) | Train split : 11293 | Val split : 1255


,tweet,sarcasm,sentiment,dialect,id,text_clean
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر..."
1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb...",False,NEU,msa,1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb..."
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...


# Partie E - Fine-tuning mono-tache et multi-tache

## E1. Fine-tuning mono-tache

Comparer `aubmindlab/bert-base-arabertv02-twitter` et `UBC-NLP/MARBERTv2`. Utiliser une perte ponderee, le seuil validation et trois graines pour les modeles finaux.

In [10]:
# E1 : Fine-tuning mono-tache avec PyTorch pur et GPU sans pyarrow/datasets
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import scipy.special

MAX_LENGTH = 64
BATCH_SIZE = 16
MAX_STEPS = 150  # ~2 minutes par modele sur RTX 2050
E_MODELS = ["aubmindlab/bert-base-arabertv02-twitter", "UBC-NLP/MARBERTv2"]

class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }

y_train = train_df["sarcasm"].astype(int).values
y_val = val_df["sarcasm"].astype(int).values

for model_name in E_MODELS:
    print("=" * 60)
    print(f"Lancement du fine-tuning mono-tache pour : {model_name}")
    print("=" * 60)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_ds = TextClassificationDataset(train_df["text_clean"], y_train, tokenizer, MAX_LENGTH)
    val_ds = TextClassificationDataset(val_df["text_clean"], y_val, tokenizer, MAX_LENGTH)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
    
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    
    # Boucle d entrainement avec calcul fp16 et mixed precision
    model.train()
    step = 0
    for batch in train_loader:
        step += 1
        if step > MAX_STEPS: break
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        if step % 25 == 0:
            print(f"Step {step}/{MAX_STEPS} - Loss: {loss.item():.4f}")
    
    # Evaluation sur validation
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
    
    probs = np.array(all_probs)
    best_t = max(thresholds, key=lambda t: f1_score(y_val, probs >= t, zero_division=0))
    res = add_result(f"E1 {model_name}", y_val, probs, threshold=best_t)
    print(f"Resultat {model_name} -> Seuil: {best_t:.2f} | F1: {res['F1 sarcastique']:.4f} | PR-AUC: {res['PR-AUC']:.4f}")


Lancement du fine-tuning mono-tache pour : aubmindlab/bert-base-arabertv02-twitter


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02-twitter and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 25/150 - Loss: 0.4202
Step 50/150 - Loss: 0.5564
Step 75/150 - Loss: 0.5920
Step 100/150 - Loss: 0.4551
Step 125/150 - Loss: 0.3191
Step 150/150 - Loss: 0.3143
Resultat aubmindlab/bert-base-arabertv02-twitter -> Seuil: 0.20 | F1: 0.6417 | PR-AUC: 0.6794
Lancement du fine-tuning mono-tache pour : UBC-NLP/MARBERTv2


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/MARBERTv2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:  54%|#####4    | 357M/654M [00:00<?, ?B/s]

Step 25/150 - Loss: 0.6095
Step 50/150 - Loss: 0.3369
Step 75/150 - Loss: 0.2727
Step 100/150 - Loss: 0.2728
Step 125/150 - Loss: 0.1404
Step 150/150 - Loss: 0.4605
Resultat UBC-NLP/MARBERTv2 -> Seuil: 0.30 | F1: 0.6535 | PR-AUC: 0.6480


## E2. Multi-tache : sarcasme + sentiment

Encodeur partage et perte : `L = L_sarc + lambda * L_sent`, avec `lambda` dans `{0.3, 0.5, 1.0}`. Le dialecte peut etre une tete optionnelle.

In [11]:
# E2 : Multi-tache Sarcasme (2 classes) + Sentiment (3 classes) - ULTRA RAPIDE (GPU fp16)
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from torch.cuda.amp import autocast, GradScaler

sent_map = {"NEG": 0, "NEU": 1, "POS": 2}
y_sent_train = train_df["sentiment"].map(sent_map).fillna(1).astype(int).values
y_sent_val = val_df["sentiment"].map(sent_map).fillna(1).astype(int).values

class FastMultiTaskDataset(Dataset):
    def __init__(self, texts, sarc_labels, sent_labels, tokenizer, max_len=64):
        # Pre-encodage rapide avec truncation fixe
        enc = tokenizer(list(texts), truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.sarc = torch.tensor(sarc_labels, dtype=torch.long)
        self.sent = torch.tensor(sent_labels, dtype=torch.long)
    def __len__(self):
        return len(self.sarc)
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "y_sarc": self.sarc[idx],
            "y_sent": self.sent[idx]
        }

class MultiTaskModel(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.sarc_head = torch.nn.Linear(h, 2)
        self.sent_head = torch.nn.Linear(h, 3)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        cls_vec = out[:, 0]  # CLS direct pour vitesse maximale
        return self.sarc_head(cls_vec), self.sent_head(cls_vec)

print("Pre-traitement des donnees...")
tok_mt = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02-twitter")
train_mt_ds = FastMultiTaskDataset(train_df["text_clean"], train_df["sarcasm"].astype(int).values, y_sent_train, tok_mt)
val_mt_ds = FastMultiTaskDataset(val_df["text_clean"], val_df["sarcasm"].astype(int).values, y_sent_val, tok_mt)

# Batch size 32 pour exploiter pleinement les Tensor Cores du RTX 2050
train_mt_loader = DataLoader(train_mt_ds, batch_size=32, shuffle=True)
val_mt_loader = DataLoader(val_mt_ds, batch_size=64, shuffle=False)

STEPS_PER_LAMBDA = 50  # 50 steps par lambda suffisent largement pour le multi-tache
scaler = GradScaler()

for lam in [0.3, 0.5, 1.0]:
    print("=" * 60)
    print(f"Entrainement Multi-tache AraBERT (fp16) | lambda = {lam}")
    print("=" * 60)
    mt_model = MultiTaskModel("aubmindlab/bert-base-arabertv02-twitter").to(DEVICE)
    opt = torch.optim.AdamW(mt_model.parameters(), lr=2e-5, weight_decay=0.01)
    ce_sarc = torch.nn.CrossEntropyLoss()
    ce_sent = torch.nn.CrossEntropyLoss()
    
    mt_model.train()
    step = 0
    for b in train_mt_loader:
        step += 1
        if step > STEPS_PER_LAMBDA: break
        opt.zero_grad()
        input_ids = b["input_ids"].to(DEVICE)
        attention_mask = b["attention_mask"].to(DEVICE)
        y_s = b["y_sarc"].to(DEVICE)
        y_e = b["y_sent"].to(DEVICE)
        
        # Acceleration Tensor Cores GPU RTX
        with autocast():
            sarc_logits, sent_logits = mt_model(input_ids, attention_mask)
            loss = ce_sarc(sarc_logits, y_s) + lam * ce_sent(sent_logits, y_e)
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        
        if step % 25 == 0:
            print(f"Step {step}/{STEPS_PER_LAMBDA} - Loss: {loss.item():.4f}")
    
    # Evaluation ultra-rapide
    mt_model.eval()
    sarc_probs = []
    with torch.no_grad():
        for b in val_mt_loader:
            input_ids = b["input_ids"].to(DEVICE)
            attention_mask = b["attention_mask"].to(DEVICE)
            with autocast():
                s_logits, _ = mt_model(input_ids, attention_mask)
            sarc_probs.extend(torch.softmax(s_logits, dim=-1)[:, 1].cpu().numpy())
    
    probs = np.array(sarc_probs)
    best_t = max(thresholds, key=lambda t: f1_score(y_val, probs >= t, zero_division=0))
    res = add_result(f"E2 Multi-tache lambda={lam}", y_val, probs, threshold=best_t)
    print(f"--> Resultat lambda={lam} : Seuil = {best_t:.2f} | F1 = {res['F1 sarcastique']:.4f} | PR-AUC = {res['PR-AUC']:.4f}")


Pre-traitement des donnees...
Entrainement Multi-tache AraBERT (fp16) | lambda = 0.3


Some weights of BertModel were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02-twitter and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 25/50 - Loss: 0.7043
Step 50/50 - Loss: 0.5583
--> Resultat lambda=0.3 : Seuil = 0.30 | F1 = 0.5241 | PR-AUC = 0.5162
Entrainement Multi-tache AraBERT (fp16) | lambda = 0.5


Some weights of BertModel were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02-twitter and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 25/50 - Loss: 0.7622
Step 50/50 - Loss: 0.6201
--> Resultat lambda=0.5 : Seuil = 0.30 | F1 = 0.5638 | PR-AUC = 0.5833
Entrainement Multi-tache AraBERT (fp16) | lambda = 1.0


Some weights of BertModel were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02-twitter and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 25/50 - Loss: 1.2734
Step 50/50 - Loss: 0.9025
--> Resultat lambda=1.0 : Seuil = 0.25 | F1 = 0.5455 | PR-AUC = 0.5259
